In [2]:
import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mount info: {e}")

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

for d in [OUTPUT_DIR, CKPT_DIR, CODE_DIR]:
    os.makedirs(d, exist_ok=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'REPO_DIR     = {REPO_DIR}')
print(f'CODE_DIR     = {CODE_DIR}')
print(f'DATA_DIR     = {DATA_DIR}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')


Drive mount info: Mountpoint must not already contain files
PROJECT_ROOT = /content/drive/MyDrive/MedThink_Verifier
REPO_DIR     = /content/drive/MyDrive/MedThink_Verifier/Medthink_repo
CODE_DIR     = /content/drive/MyDrive/MedThink_Verifier/Medthink_repo/Medthink_Code
DATA_DIR     = /content/drive/MyDrive/MedThink_Verifier/Medthink_repo/Medthink_Dataset
OUTPUT_DIR   = /content/drive/MyDrive/MedThink_Verifier/phase1_predictions


In [3]:
import torch

print("=== 1. KIỂM TRA GPU ===")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {device_name}")
    print(f"   VRAM: {vram_gb:.1f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy đổi Runtime sang GPU T4.")


=== 1. KIỂM TRA GPU ===
✅ GPU: Tesla T4
   VRAM: 15.6 GB


In [4]:
# ============================================================
# Cell 2: Self-Healed Visual Feature Extraction for R-RAD & R-SLAKE
# ============================================================
import os, sys, json
import torch
import torchvision.transforms as T
from PIL import Image
from tqdm.notebook import tqdm

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

os.makedirs(f'{PROJECT_ROOT}/features/R-RAD', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/features/R-SLAKE', exist_ok=True)

rad_img_dir   = f'{DATA_DIR}/R-RAD/images'
slake_img_dir = f'{DATA_DIR}/R-SLAKE/images'

if not os.path.exists(rad_img_dir) or len(os.listdir(rad_img_dir) if os.path.exists(rad_img_dir) else []) == 0:
    print("🔄 Fetching dataset images via Git LFS...")
    !apt-get install -y git-lfs -q
    !git lfs install
    tmp_clone = '/content/medthink_tmp'
    !rm -rf {tmp_clone}
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/Tang-xiaoxiao/Medthink.git {tmp_clone}
    !cd {tmp_clone} && git lfs pull --include="Medthink_Dataset/**"
    os.makedirs(REPO_DIR, exist_ok=True)
    !cp -r {tmp_clone}/Medthink_Dataset {REPO_DIR}/
    !rm -rf {tmp_clone}
    print("✅ Dataset images fetched successfully!")

detr_dir = f'{PROJECT_ROOT}/detr_cooelf'
if not os.path.exists(f'{detr_dir}/hubconf.py'):
    print('🔄 Cloning cooelf/detr fork...')
    !git clone --depth 1 https://github.com/cooelf/detr.git {detr_dir}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print('🔄 Loading DETR ResNet-101 model tu cooelf/detr...')
detr_model = torch.hub.load(detr_dir, 'detr_resnet101', source='local', pretrained=True)
detr_model.to(device)
detr_model.eval()

transform = T.Compose([
    T.Resize((800, 800)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_features_for_dataset(ds_name, img_dir, out_pth, out_map):
    img_files = []
    if os.path.exists(img_dir):
        for root, dirs, files in os.walk(img_dir):
            for f in files:
                if f.endswith(('.jpg', '.png', '.jpeg')):
                    img_files.append(os.path.relpath(os.path.join(root, f), img_dir))

    if os.path.exists(out_pth) and os.path.getsize(out_pth) > 1000 and os.path.exists(out_map):
        print(f'✅ Features cho {ds_name} da ton tai tai {out_pth}')
        name_map = {}
        for idx, fname in enumerate(sorted(img_files)):
            bname = os.path.basename(fname)
            no_ext = os.path.splitext(bname)[0]
            name_map[fname] = idx
            name_map[bname] = idx
            name_map[no_ext] = idx
            name_map[str(no_ext)] = idx
            parts = fname.replace('\\', '/').split('/')
            if len(parts) > 1:
                name_map[parts[0]] = idx
                name_map[parts[-1]] = idx
                name_map[os.path.splitext(parts[-1])[0]] = idx
        with open(out_map, 'w') as f:
            json.dump(name_map, f, indent=2)
        print(f'✅ Updated {out_map} with multi-key mappings ({len(name_map)} keys)')
        return

    print(f'🔄 Trich xuat DETR features cho {ds_name}...')
    if not os.path.exists(img_dir):
        print(f'❌ Thu muc anh khong ton tai: {img_dir}')
        return

    print(f'Tim thấy {len(img_files)} ảnh trong {ds_name}')

    name_map = {}
    feats_list = []

    with torch.no_grad():
        for idx, fname in enumerate(tqdm(sorted(img_files), desc=f'Extracting {ds_name}')):
            fpath = os.path.join(img_dir, fname)
            try:
                img = Image.open(fpath).convert('RGB')
                tensor_img = transform(img).unsqueeze(0).to(device)
                outputs = detr_model(tensor_img)
                if isinstance(outputs, dict) and 'hs' in outputs:
                    last_hs = outputs['hs'][-1].squeeze(0)
                elif hasattr(outputs, 'last_hidden_state'):
                    last_hs = outputs.last_hidden_state.squeeze(0)
                else:
                    last_hs = outputs[0][-1].squeeze(0) if isinstance(outputs, (list, tuple)) else outputs

                feats_list.append(last_hs.cpu())
                bname = os.path.basename(fname)
                no_ext = os.path.splitext(bname)[0]
                name_map[fname] = idx
                name_map[bname] = idx
                name_map[no_ext] = idx
                name_map[str(no_ext)] = idx
                parts = fname.replace('\\', '/').split('/')
                if len(parts) > 1:
                    name_map[parts[0]] = idx
                    name_map[parts[-1]] = idx
                    name_map[os.path.splitext(parts[-1])[0]] = idx
            except Exception as e:
                print(f'Warning: Error image {fname}: {e}')

    if len(feats_list) > 0:
        stacked_feats = torch.stack(feats_list)
        torch.save(stacked_feats, out_pth)
        with open(out_map, 'w') as f:
            json.dump(name_map, f, indent=2)
        print(f'✅ Da luu {stacked_feats.shape} features tai {out_pth}')
    else:
        print(f'❌ Khong trich xuat duoc feature nao cho {ds_name}')

extract_features_for_dataset(
    'rad',
    rad_img_dir,
    f'{PROJECT_ROOT}/features/R-RAD/detr.pth',
    f'{PROJECT_ROOT}/features/R-RAD/name_map.json'
)

extract_features_for_dataset(
    'slake',
    slake_img_dir,
    f'{PROJECT_ROOT}/features/R-SLAKE/detr.pth',
    f'{PROJECT_ROOT}/features/R-SLAKE/name_map.json'
)


🔄 Fetching dataset images via Git LFS...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 1s (5,030 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into '/content/medthink_tmp'...
remote: Enumerating objects: 8452, done.
remote: Counting objects: 100% (8452/8452), done.
remote: Compressing objects: 100% (8

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 184MB/s]


Downloading: "https://dl.fbaipublicfiles.com/detr/detr-r101-2c7b67e5.pth" to /root/.cache/torch/hub/checkpoints/detr-r101-2c7b67e5.pth


...

🔄 Trich xuat DETR features cho rad...
Tim thấy 315 ảnh trong rad


Extracting rad:   0%|          | 0/315 [00:00<?, ?it/s]

✅ Da luu torch.Size([315, 6, 1, 100, 256]) features tai /content/drive/MyDrive/MedThink_Verifier/features/R-RAD/detr.pth
🔄 Trich xuat DETR features cho slake...
Tim thấy 1284 ảnh trong slake


Extracting slake:   0%|          | 0/1284 [00:00<?, ?it/s]

✅ Da luu torch.Size([1284, 6, 1, 100, 256]) features tai /content/drive/MyDrive/MedThink_Verifier/features/R-SLAKE/detr.pth


In [5]:
# ============================================================
# Cell 3: Multi-Epoch Training Pipeline (5 Epochs for R-RAD Reasoning)
# ============================================================
import os, sys, json, importlib
import torch

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

for d in [OUTPUT_DIR, CKPT_DIR, CODE_DIR]:
    os.makedirs(d, exist_ok=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# 1. Write dataset.py directly to CODE_DIR
dataset_py_path = f"{CODE_DIR}/dataset.py"
dataset_py_code = r"""import json, os
import torch
from torch.utils.data import Dataset

class OpenMedVQADataset(Dataset):
    def __init__(self, _tokenizer, _text_file_path, _img_file_path, _img_name_map,
                 _method, _dataset, _source_len, _target_len):
        self.tokenizer = _tokenizer
        self.source_text = []
        self.target_text = []
        self.problem_id = []
        self.img_index = []
        self.pretrained_feature = torch.load(_img_file_path)
        self.source_len = _source_len
        self.target_len = _target_len

        with open(_text_file_path, "r", encoding="utf-8") as TextFile:
            data = json.load(TextFile)
        with open(_img_name_map, "r", encoding="utf-8") as NameFile:
            name_map = json.load(NameFile)

        for problem in data:
            pair = OpenInputAndTargetAndImg(data[problem])
            prompt = pair.get_input(method=_method)
            target = pair.get_target(method=_method)
            img = pair.get_img(_dataset=_dataset)
            self.source_text.append(prompt)
            self.target_text.append(target)
            img_key = str(img)
            if img_key not in name_map:
                if f"{img_key}.jpg" in name_map:
                    img_key = f"{img_key}.jpg"
                elif f"{img_key}.png" in name_map:
                    img_key = f"{img_key}.png"
                elif os.path.splitext(img_key)[0] in name_map:
                    img_key = os.path.splitext(img_key)[0]
                else:
                    img_key = list(name_map.keys())[0]
            self.img_index.append(int(name_map[img_key]))
            self.problem_id.append(problem)

    def __len__(self):
        return len(self.source_text)

    def __getitem__(self, item):
        source_text = str(self.source_text[item])
        target_text = str(self.target_text[item])
        img_index = self.img_index[item]

        source_text = " ".join(source_text.split())
        target_text = " ".join(target_text.split())

        source = self.tokenizer(
            source_text,
            max_length=self.source_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target = self.tokenizer(
            target_text,
            max_length=self.target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        source_ids = source["input_ids"].squeeze()
        source_mask = source["attention_mask"].squeeze()
        image_ids = self.pretrained_feature[img_index].squeeze()
        target_ids = target["input_ids"].squeeze()

        return {
            "input_ids": source_ids,
            "attention_mask": source_mask,
            "image_ids": image_ids,
            "labels": target_ids
        }

class OpenInputAndTargetAndImg:
    def __init__(self, problem):
        self.problem = problem
        self.question_text = self.get_question_text()
        self.answer_text = self.get_answer()
        self.solution_text = self.get_solution_text()

    def get_question_text(self):
        return self.problem['question']

    def get_answer(self):
        return self.problem['choices'][0]

    def get_solution_text(self):
        return self.problem['solution']

    def get_target(self, method):
        if method == "Explanation":
            return f"The answer is {self.answer_text}.\nSolution: {self.problem['solution']}"
        elif method == "Reasoning":
            return f"{self.problem['solution']}\nAnswer: The answer is {self.answer_text}."
        elif method == "First-Stage_Reasoning":
            return f"{self.problem['solution']}"
        elif method == "Second-Stage_Reasoning":
            return f"The answer is {self.answer_text}."
        elif method == "without_R":
            return f"The answer is {self.answer_text}."
        else:
            raise ValueError("Please Check the Value of Method.")

    def get_input(self, method):
        if method == "Explanation":
            return f"Question: {self.question_text}\nAnswer:"
        elif method == "Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "First-Stage_Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "Second-Stage_Reasoning":
            return f"Question: {self.question_text}\nSolution: {self.solution_text}\nAnswer:"
        else:
            raise ValueError("Please Check the Value of Method.")

    def get_img(self, _dataset):
        if _dataset == "rad":
            return self.problem['image'][:-4]
        elif _dataset == "slake":
            return str(self.problem["img_id"])
        else:
            raise ValueError(f"Invalid _dataset value: {_dataset}. The value must be 'rad' or 'slake'.")
"""

with open(dataset_py_path, 'w', encoding='utf-8') as f:
    f.write(dataset_py_code)

# 2. Write model.py directly to CODE_DIR
model_py_path = f"{CODE_DIR}/model.py"
model_py_code = r"""from transformers import T5Config, T5ForConditionalGeneration
try:
    from transformers.models.t5.modeling_t5 import T5Stack, __HEAD_MASK_WARNING_MSG, T5Block, T5LayerNorm
except ImportError:
    from transformers.models.t5.modeling_t5 import T5Stack, T5Block, T5LayerNorm
    __HEAD_MASK_WARNING_MSG = "head_mask warning"
import copy
from transformers.modeling_outputs import ModelOutput, BaseModelOutput, BaseModelOutputWithPast, \
    BaseModelOutputWithPastAndCrossAttentions, Seq2SeqLMOutput, Seq2SeqModelOutput
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple, Union
import math
import os
import warnings
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import (
    BaseModelOutput,
    Seq2SeqLMOutput,
)
try:
    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map
except ImportError:
    assert_device_map, get_device_map = None, None
from torch.utils.checkpoint import checkpoint


class JointEncoder(T5Stack):
    def __init__(self, config, embed_tokens=None, patch_size=None):
        super().__init__(config)

        self.embed_tokens = embed_tokens
        self.is_decoder = config.is_decoder

        self.patch_num, self.patch_dim = patch_size
        self.image_dense = nn.Linear(self.patch_dim, config.d_model)
        self.mha_layer = torch.nn.MultiheadAttention(embed_dim=config.hidden_size, kdim=config.hidden_size,
                                                     vdim=config.hidden_size, num_heads=1, batch_first=True)
        self.gate_dense = nn.Linear(2 * config.hidden_size, config.hidden_size)
        self.sigmoid = nn.Sigmoid()

        self.block = nn.ModuleList(
            [T5Block(config, has_relative_attention_bias=bool(i == 0)) for i in range(config.num_layers)]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

        self.post_init()
        self.model_parallel = False
        self.device_map = None
        self.gradient_checkpointing = False

    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = getattr(self, 'dtype', torch.float32)
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

    def invert_attention_mask(self, encoder_attention_mask):
        if encoder_attention_mask.dim() == 3:
            encoder_extended_attention_mask = encoder_attention_mask[:, None, :, :]
        elif encoder_attention_mask.dim() == 2:
            encoder_extended_attention_mask = encoder_attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for encoder_attention_mask: {encoder_attention_mask.shape}")
        dtype = getattr(self, 'dtype', torch.float32)
        encoder_extended_attention_mask = encoder_extended_attention_mask.to(dtype=dtype)
        encoder_extended_attention_mask = (1.0 - encoder_extended_attention_mask) * torch.finfo(dtype).min
        return encoder_extended_attention_mask

    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            dtype = getattr(self, 'dtype', torch.float32)
            head_mask = head_mask.to(dtype=dtype)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def forward(
            self,
            input_ids=None,
            attention_mask=None,
            encoder_hidden_states=None,
            encoder_attention_mask=None,
            inputs_embeds=None,
            image_ids=None,
            head_mask=None,
            cross_attn_head_mask=None,
            past_key_values=None,
            use_cache=None,
            output_attentions=None,
            output_hidden_states=None,
            return_dict=None,
    ):
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None and inputs_embeds is not None:
            err_msg_prefix = "decoder_" if self.is_decoder else ""
            raise ValueError(
                f"You cannot specify both {err_msg_prefix}input_ids and {err_msg_prefix}inputs_embeds at the same time"
            )
        elif input_ids is not None:
            input_shape = input_ids.size()
            input_ids = input_ids.view(-1, input_shape[-1])
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]
        else:
            err_msg_prefix = "decoder_" if self.is_decoder else ""
            raise ValueError(f"You have to specify either {err_msg_prefix}input_ids or {err_msg_prefix}inputs_embeds")

        if inputs_embeds is None:
            assert self.embed_tokens is not None, "You have to initialize the model with valid token embeddings"
            inputs_embeds = self.embed_tokens(input_ids)

        batch_size, seq_length = input_shape
        if past_key_values is not None:
            if hasattr(past_key_values, "get_seq_length"):
                past_length = past_key_values.get_seq_length()
            elif isinstance(past_key_values, (tuple, list)) and len(past_key_values) > 0 and past_key_values[0] is not None:
                past_length = past_key_values[0][0].shape[2]
            else:
                past_length = 0
            mask_seq_length = past_length + seq_length
        else:
            mask_seq_length = seq_length

        if use_cache is True:
            assert self.is_decoder, f"`use_cache` can only be set to `True` if {self} is used as a decoder"

        if attention_mask is None:
            attention_mask = torch.ones(batch_size, mask_seq_length, device=inputs_embeds.device)
        if self.is_decoder and encoder_attention_mask is None and encoder_hidden_states is not None:
            encoder_seq_length = encoder_hidden_states.shape[1]
            encoder_attention_mask = torch.ones(
                batch_size, encoder_seq_length, device=inputs_embeds.device, dtype=torch.long
            )

        extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)

        if self.is_decoder and encoder_hidden_states is not None:
            encoder_batch_size, encoder_sequence_length, _ = encoder_hidden_states.size()
            encoder_hidden_shape = (encoder_batch_size, encoder_sequence_length)
            if encoder_attention_mask is None:
                encoder_attention_mask = torch.ones(encoder_hidden_shape, device=inputs_embeds.device)
            encoder_extended_attention_mask = self.invert_attention_mask(encoder_attention_mask)
        else:
            encoder_extended_attention_mask = None

        head_mask = self.get_head_mask(head_mask, self.config.num_layers)
        cross_attn_head_mask = self.get_head_mask(cross_attn_head_mask, self.config.num_layers)
        present_key_value_states = () if use_cache else None
        all_hidden_states = () if output_hidden_states else None
        all_attentions = () if output_attentions else None
        all_cross_attentions = () if (output_attentions and self.is_decoder) else None
        position_bias = None
        encoder_decoder_position_bias = None

        if past_key_values is None or not isinstance(past_key_values, (tuple, list)):
            past_key_values_loop = [None] * len(self.block)
        else:
            past_key_values_loop = past_key_values

        hidden_states = self.dropout(inputs_embeds)

        for i, (layer_module, past_key_value) in enumerate(zip(self.block, past_key_values_loop)):
            layer_head_mask = head_mask[i]
            cross_attn_layer_head_mask = cross_attn_head_mask[i]
            if output_hidden_states:
                all_hidden_states = all_hidden_states + (hidden_states,)

            layer_outputs = layer_module(
                hidden_states,
                attention_mask=extended_attention_mask,
                position_bias=position_bias,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=encoder_extended_attention_mask,
                encoder_decoder_position_bias=encoder_decoder_position_bias,
                layer_head_mask=layer_head_mask,
                cross_attn_layer_head_mask=cross_attn_layer_head_mask,
                past_key_value=past_key_value,
                use_cache=use_cache,
                output_attentions=output_attentions,
            )

            if use_cache is False:
                layer_outputs = layer_outputs[:1] + (None,) + layer_outputs[1:]

            hidden_states, present_key_value_state = layer_outputs[:2]
            position_bias = layer_outputs[2]
            if self.is_decoder and encoder_hidden_states is not None:
                encoder_decoder_position_bias = layer_outputs[4 if output_attentions else 3]
            if use_cache:
                present_key_value_states = present_key_value_states + (present_key_value_state,)

            if output_attentions:
                all_attentions = all_attentions + (layer_outputs[3],)
                if self.is_decoder:
                    all_cross_attentions = all_cross_attentions + (layer_outputs[5],)

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)

        if output_hidden_states:
            all_hidden_states = all_hidden_states + (hidden_states,)

        if image_ids is not None:
            if image_ids.dim() > 3:
                image_ids = image_ids.view(image_ids.size(0), -1, image_ids.size(-1))
            image_embedding = self.image_dense(image_ids)
            if image_embedding.dim() > 3:
                image_embedding = image_embedding.view(image_embedding.size(0), -1, image_embedding.size(-1))
            image_att, _ = self.mha_layer(hidden_states, image_embedding, image_embedding)
            merge = torch.cat([hidden_states, image_att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))
            hidden_states = (1 - gate) * hidden_states + gate * image_att

        return BaseModelOutputWithPastAndCrossAttentions(
            last_hidden_state=hidden_states,
            past_key_values=present_key_value_states,
            hidden_states=all_hidden_states,
            attentions=all_attentions,
            cross_attentions=all_cross_attentions,
        )


class T5ForMultimodalGeneration(T5ForConditionalGeneration):
    def __init__(self, config, patch_size=(100, 256)):
        super().__init__(config)
        self.model_dim = config.d_model

        self.shared = nn.Embedding(config.vocab_size, config.d_model)

        encoder_config = copy.deepcopy(config)
        encoder_config.is_decoder = False
        encoder_config.use_cache = False
        encoder_config.is_encoder_decoder = False
        self.encoder = JointEncoder(encoder_config, patch_size=patch_size)
        self.encoder.embed_tokens = self.shared

        decoder_config = copy.deepcopy(config)
        decoder_config.is_decoder = True
        decoder_config.is_encoder_decoder = False
        decoder_config.num_layers = config.num_decoder_layers
        self.decoder = T5Stack(decoder_config)
        self.decoder.embed_tokens = self.shared

        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.post_init()

    def forward(
            self,
            input_ids: Optional[torch.LongTensor] = None,
            image_ids: Optional[torch.FloatTensor] = None,
            attention_mask: Optional[torch.FloatTensor] = None,
            decoder_input_ids: Optional[torch.LongTensor] = None,
            decoder_attention_mask: Optional[torch.BoolTensor] = None,
            head_mask: Optional[torch.FloatTensor] = None,
            decoder_head_mask: Optional[torch.FloatTensor] = None,
            cross_attn_head_mask: Optional[torch.FloatTensor] = None,
            encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
            past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
            inputs_embeds: Optional[torch.FloatTensor] = None,
            decoder_inputs_embeds: Optional[torch.FloatTensor] = None,
            labels: Optional[torch.LongTensor] = None,
            use_cache: Optional[bool] = None,
            output_attentions: Optional[bool] = None,
            output_hidden_states: Optional[bool] = None,
            return_dict: Optional[bool] = None,
    ) -> Union[Tuple[torch.FloatTensor], Seq2SeqLMOutput]:
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
                inputs_embeds=inputs_embeds,
                head_mask=head_mask,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )

        hidden_states = encoder_outputs[0]

        if labels is not None and decoder_input_ids is None and decoder_inputs_embeds is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            inputs_embeds=decoder_inputs_embeds,
            past_key_values=past_key_values,
            encoder_hidden_states=hidden_states,
            encoder_attention_mask=attention_mask,
            head_mask=decoder_head_mask,
            cross_attn_head_mask=cross_attn_head_mask,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = decoder_outputs[0]

        if self.config.tie_word_embeddings:
            sequence_output = sequence_output * (self.model_dim ** -0.5)

        lm_logits = self.lm_head(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

        if not return_dict:
            output = (lm_logits,) + decoder_outputs[1:] + encoder_outputs
            return ((loss,) + output) if loss is not None else output

        return Seq2SeqLMOutput(
            loss=loss,
            logits=lm_logits,
            past_key_values=decoder_outputs.past_key_values,
            decoder_hidden_states=decoder_outputs.hidden_states,
            decoder_attentions=decoder_outputs.attentions,
            cross_attentions=decoder_outputs.cross_attentions,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
            encoder_hidden_states=encoder_outputs.hidden_states,
            encoder_attentions=encoder_outputs.attentions,
        )

    def prepare_inputs_for_generation(
            self,
            input_ids,
            past_key_values=None,
            attention_mask=None,
            head_mask=None,
            decoder_head_mask=None,
            cross_attn_head_mask=None,
            use_cache=None,
            encoder_outputs=None,
            **kwargs,
    ):
        if past_key_values is not None:
            if hasattr(past_key_values, "get_seq_length"):
                past_length = past_key_values.get_seq_length()
            elif isinstance(past_key_values, (tuple, list)) and len(past_key_values) > 0 and past_key_values[0] is not None:
                past_length = past_key_values[0][0].shape[2]
            else:
                past_length = 0

            if past_length > 0:
                if input_ids.shape[1] > past_length:
                    remove_prefix_length = past_length
                else:
                    remove_prefix_length = input_ids.shape[1] - 1
                input_ids = input_ids[:, remove_prefix_length:]

        image_ids = kwargs.get("image_ids", None)

        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "decoder_head_mask": decoder_head_mask,
            "cross_attn_head_mask": cross_attn_head_mask,
            "use_cache": use_cache,
            "image_ids": image_ids,
        }
"""

with open(model_py_path, 'w', encoding='utf-8') as f:
    f.write(model_py_code)

print("✅ Successfully updated dataset.py and model.py directly in CODE_DIR")

if 'model' in sys.modules:
    importlib.reload(sys.modules['model'])
if 'dataset' in sys.modules:
    importlib.reload(sys.modules['dataset'])

from dataset import OpenMedVQADataset
from model import T5ForMultimodalGeneration

TRAIN_CONFIG = [
    (
        'rad', 'Reasoning', 5,
        f'{DATA_DIR}/R-RAD/open-end/trainset.json',
        f'{PROJECT_ROOT}/features/R-RAD/detr.pth',
        f'{PROJECT_ROOT}/features/R-RAD/name_map.json',
    ),
]

import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoTokenizer

base_model_name = 'allenai/unifiedqa-t5-base'
print(f'Loading tokenizer: {base_model_name}')
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

for ds_name, method, num_epochs, train_json, detr_pth, name_map_json in TRAIN_CONFIG:
    print(f'\n============================================================')
    print(f'🚀 Running Multi-Epoch Training ({num_epochs} Epochs) for {ds_name} | {method}')
    print(f'============================================================')

    ckpt_out_dir = f'{CKPT_DIR}/{ds_name}_{method}'
    os.makedirs(ckpt_out_dir, exist_ok=True)

    print(f'Loading model from: {base_model_name}')
    model_obj = T5ForMultimodalGeneration.from_pretrained(
        base_model_name,
        patch_size=(100, 256)
    )

    print(f'Loading dataset: {train_json}')
    train_dataset = OpenMedVQADataset(
        _tokenizer=tokenizer,
        _text_file_path=train_json,
        _img_file_path=detr_pth,
        _img_name_map=name_map_json,
        _method=method,
        _dataset=ds_name,
        _source_len=512,
        _target_len=256
    )
    print(f'Train set size: {len(train_dataset)} samples')

    training_args = Seq2SeqTrainingArguments(
        output_dir=ckpt_out_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        weight_decay=0.01,
        save_strategy='epoch',
        eval_strategy='no',
        logging_steps=10,
        fp16=torch.cuda.is_available(),
        predict_with_generate=True,
        report_to='none',
        remove_unused_columns=False
    )

    trainer = Seq2SeqTrainer(
        model=model_obj,
        args=training_args,
        train_dataset=train_dataset,
        processing_class=tokenizer
    )

    print(f'🚀 Starting training {ds_name} | {method} for {num_epochs} epochs...')
    train_result = trainer.train()
    trainer.save_model(ckpt_out_dir)
    print(f'✅ Multi-Epoch Training complete! Saved final checkpoint to {ckpt_out_dir}')


✅ Successfully updated dataset.py and model.py directly in CODE_DIR
Loading tokenizer: allenai/unifiedqa-t5-base


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Multi-Epoch Training complete! Saved final checkpoint to /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_Reasoning


In [6]:
# ============================================================
# Cell 4: Full Batch Inference Pipeline (Phase 1.2 Multi-Epoch)
# ============================================================
import os, sys, json, re, importlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import transformers
from transformers import AutoTokenizer
import transformers.integrations.sdpa_attention as sdpa_attn

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# 1. SDPA attention mask auto-padding patch
if not hasattr(sdpa_attn, '_orig_create_position_bias_mask'):
    sdpa_attn._orig_create_position_bias_mask = sdpa_attn.create_position_bias_mask

def safe_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key):
    if attention_mask is not None and position_bias is not None:
        if attention_mask.shape[-1] < position_bias.shape[-1]:
            diff = position_bias.shape[-1] - attention_mask.shape[-1]
            pad = torch.ones((*attention_mask.shape[:-1], diff), dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([attention_mask, pad], dim=-1)
        elif attention_mask.shape[-1] > position_bias.shape[-1]:
            attention_mask = attention_mask[..., :position_bias.shape[-1]]
    return sdpa_attn._orig_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key)

sdpa_attn.create_position_bias_mask = safe_create_position_bias_mask
print("✅ Patched sdpa_attention.create_position_bias_mask")

from dataset import OpenMedVQADataset
from model import T5ForMultimodalGeneration

def parse_reasoning_output(text):
    text = text.strip()
    if 'Answer:' in text:
        parts = text.split('Answer:', 1)
        rationale = parts[0].strip()
        answer_part = parts[1].strip()
    elif '\n' in text:
        lines = text.rsplit('\n', 1)
        rationale = lines[0].strip()
        answer_part = lines[1].strip()
    else:
        rationale = ''
        answer_part = text

    m = re.search(r'[Tt]he answer is (.+?)[.!]?$', answer_part)
    answer = m.group(1).strip() if m else answer_part.strip()
    return rationale, answer

def run_inference_rad_reasoning():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device for inference: {device}")

    base_model_name = 'allenai/unifiedqa-t5-base'
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    ckpt_dir = f"{CKPT_DIR}/rad_Reasoning"
    if os.path.exists(f"{ckpt_dir}/checkpoint-780"):
        load_path = f"{ckpt_dir}/checkpoint-780"
    elif os.path.exists(f"{ckpt_dir}/checkpoint-156"):
        load_path = f"{ckpt_dir}/checkpoint-156"
    elif os.path.exists(ckpt_dir) and os.path.exists(f"{ckpt_dir}/config.json"):
        load_path = ckpt_dir
    else:
        print(f"⚠️ Checkpoint not found at {ckpt_dir}. Loading pre-trained base model weights {base_model_name}...")
        load_path = base_model_name

    print(f"Loading trained 5-epoch model from: {load_path}")
    eval_model = T5ForMultimodalGeneration.from_pretrained(
        load_path,
        patch_size=(100, 256)
    )
    eval_model.to(device)
    eval_model.eval()

    test_json = f"{DATA_DIR}/R-RAD/open-end/testset.json"
    detr_pth = f"{PROJECT_ROOT}/features/R-RAD/detr.pth"
    name_map_json = f"{PROJECT_ROOT}/features/R-RAD/name_map.json"

    print(f"Loading test dataset: {test_json}")
    test_dataset = OpenMedVQADataset(
        _tokenizer=tokenizer,
        _text_file_path=test_json,
        _img_file_path=detr_pth,
        _img_name_map=name_map_json,
        _method="Reasoning",
        _dataset="rad",
        _source_len=512,
        _target_len=256
    )
    print(f"Test dataset size: {len(test_dataset)} samples")

    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    results = []
    with open(test_json, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    prob_ids = list(raw_data.keys())

    print("🚀 Running full batch inference on test set (5-Epoch Model)...")
    idx = 0
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inference"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image_ids = batch['image_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = eval_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
                max_length=256,
                num_beams=2,
                early_stopping=True
            )

            decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            for pred_str, label_str in zip(decoded_preds, decoded_labels):
                pid = prob_ids[idx]
                item = raw_data[pid]
                gt_ans = item['choices'][0]
                gt_rat = item.get('solution', '')

                pred_rat, pred_ans = parse_reasoning_output(pred_str)

                results.append({
                    'problem_id': pid,
                    'question': item['question'],
                    'gt_answer': gt_ans,
                    'gt_rationale': gt_rat,
                    'raw_prediction': pred_str,
                    'pred_answer': pred_ans,
                    'pred_rationale': pred_rat
                })
                idx += 1

    df = pd.DataFrame(results)
    out_csv = f"{OUTPUT_DIR}/predictions_rad_Reasoning_5epochs.csv"
    df.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"✅ Phase 1.2 Inference complete! Saved {len(df)} predictions to {out_csv}")
    print(df.head(3))
    return df

df_preds = run_inference_rad_reasoning()


✅ Patched sdpa_attention.create_position_bias_mask
Using device for inference: cuda
Loading trained 5-epoch model from: /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_Reasoning/checkpoint-780


Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

Loading test dataset: /content/drive/MyDrive/MedThink_Verifier/Medthink_repo/Medthink_Dataset/R-RAD/open-end/testset.json
Test dataset size: 179 samples
🚀 Running full batch inference on test set (5-Epoch Model)...


Inference:   0%|          | 0/23 [00:00<?, ?it/s]

✅ Phase 1.2 Inference complete! Saved 179 predictions to /content/drive/MyDrive/MedThink_Verifier/phase1_predictions/predictions_rad_Reasoning_5epochs.csv
  problem_id                                       question  \
0         20                   How is the patient oriented?   
1        302               Describe the lung abnormalities?   
2        304  What abnormalities are seen within the lungs?   

            gt_answer                                       gt_rationale  \
0  Posterior-Anterior  The orientation of the patient for this X-ray ...   
1   Pulmonary nodules  This chest x-ray demonstrates several rounded ...   
2   Pulmonary nodules  Pulmonary nodules can be identified on a chest...   

                                      raw_prediction  \
0  dispozitie dispozitie dispozitie dispozitie di...   
1  dispozitiescreenscreen dispozitie dispozitie d...   
2  dispozitie Cinema dispozitie dispozitie dispoz...   

                                         pred_answer pred_rati

In [7]:
# ============================================================
# Cell 5: Quantitative Evaluation Metrics (Phase 1.2 Multi-Epoch)
# ============================================================
import os, sys, json, re
import pandas as pd
import numpy as np

# Install evaluation packages if needed
!pip install -q rouge-score nltk bert-score evaluate

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
csv_path     = f"{OUTPUT_DIR}/predictions_rad_Reasoning_5epochs.csv"

if not os.path.exists(csv_path):
    print(f"❌ Prediction file not found at {csv_path}")
else:
    df = pd.read_csv(csv_path)
    print(f"✅ Loaded {len(df)} predictions from {csv_path}")

    # 1. Compute Answer Accuracy (Exact Match & Normalized Match)
    exact_matches = 0
    norm_matches = 0

    def normalize_str(s):
        if not isinstance(s, str):
            return ""
        s = s.lower().strip()
        s = re.sub(r'[^\w\s]', '', s)
        return s

    for _, row in df.iterrows():
        gt_ans = str(row['gt_answer'])
        pred_ans = str(row['pred_answer'])

        if gt_ans.strip().lower() == pred_ans.strip().lower():
            exact_matches += 1

        if normalize_str(gt_ans) in normalize_str(pred_ans) or normalize_str(pred_ans) in normalize_str(gt_ans):
            norm_matches += 1

    acc_exact = exact_matches / len(df) * 100
    acc_norm  = norm_matches / len(df) * 100

    # 2. Compute ROUGE Scores (Rationale & Full Prediction)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1_list, r2_list, rl_list = [], [], []

    for _, row in df.iterrows():
        ref = str(row['gt_rationale']) if pd.notna(row['gt_rationale']) and str(row['gt_rationale']).strip() else str(row['gt_answer'])
        hyp = str(row['pred_rationale']) if pd.notna(row['pred_rationale']) and str(row['pred_rationale']).strip() else str(row['raw_prediction'])

        scores = scorer.score(ref, hyp)
        r1_list.append(scores['rouge1'].fmeasure)
        r2_list.append(scores['rouge2'].fmeasure)
        rl_list.append(scores['rougeL'].fmeasure)

    r1 = np.mean(r1_list) * 100
    r2 = np.mean(r2_list) * 100
    rl = np.mean(rl_list) * 100

    # 3. Compute BLEU Scores
    smooth = SmoothingFunction().method1
    b1_list, b2_list, b3_list, b4_list = [], [], [], []

    for _, row in df.iterrows():
        ref_tokens = [nltk.word_tokenize(str(row['gt_rationale']).lower())] if pd.notna(row['gt_rationale']) else [[str(row['gt_answer']).lower()]]
        hyp_tokens = nltk.word_tokenize(str(row['raw_prediction']).lower())

        b1_list.append(sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth))
        b2_list.append(sentence_bleu(ref_tokens, hyp_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth))
        b3_list.append(sentence_bleu(ref_tokens, hyp_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth))
        b4_list.append(sentence_bleu(ref_tokens, hyp_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth))

    b1 = np.mean(b1_list) * 100
    b2 = np.mean(b2_list) * 100
    b3 = np.mean(b3_list) * 100
    b4 = np.mean(b4_list) * 100

    # 4. Try BERTScore calculation
    bert_f1 = None
    try:
        from bert_score import score as bert_score_func
        cands = df['raw_prediction'].fillna("").astype(str).tolist()
        refs  = df['gt_rationale'].fillna(df['gt_answer']).astype(str).tolist()
        P, R, F1 = bert_score_func(cands, refs, lang="en", verbose=False)
        bert_f1 = F1.mean().item() * 100
    except Exception as e:
        print(f"Note: BERTScore computation skipped ({e})")

    metrics = {
        'Dataset': 'R-RAD',
        'Method': 'Reasoning',
        'Epochs': 5,
        'Samples': len(df),
        'Accuracy (Exact)': f"{acc_exact:.2f}%",
        'Accuracy (Norm)': f"{acc_norm:.2f}%",
        'ROUGE-1': f"{r1:.2f}",
        'ROUGE-2': f"{r2:.2f}",
        'ROUGE-L': f"{rl:.2f}",
        'BLEU-1': f"{b1:.2f}",
        'BLEU-2': f"{b2:.2f}",
        'BLEU-3': f"{b3:.2f}",
        'BLEU-4': f"{b4:.2f}",
    }
    if bert_f1 is not None:
        metrics['BERTScore F1'] = f"{bert_f1:.2f}"

    print("\n============================================================")
    print("📊 PHASE 1.2 EVALUATION METRICS (5 EPOCHS - R-RAD REASONING)")
    print("============================================================")
    metrics_df = pd.DataFrame([metrics])
    print(metrics_df.to_string(index=False))

    metrics_out = f"{OUTPUT_DIR}/metrics_rad_Reasoning_5epochs.json"
    with open(metrics_out, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\n✅ Metrics saved to {metrics_out}")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
✅ Loaded 179 predictions from /content/drive/MyDrive/MedThink_Verifier/phase1_predictions/predictions_rad_Reasoning_5epochs.csv


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 PHASE 1.2 EVALUATION METRICS (5 EPOCHS - R-RAD REASONING)
Dataset    Method  Epochs  Samples Accuracy (Exact) Accuracy (Norm) ROUGE-1 ROUGE-2 ROUGE-L BLEU-1 BLEU-2 BLEU-3 BLEU-4 BERTScore F1
  R-RAD Reasoning       5      179            0.00%           0.00%    0.00    0.00    0.00   0.00   0.00   0.00   0.00        69.79

✅ Metrics saved to /content/drive/MyDrive/MedThink_Verifier/phase1_predictions/metrics_rad_Reasoning_5epochs.json


In [4]:
# ============================================================
# Cell 6: Self-Healed Stage 1 Training (First-Stage_Reasoning, LR=5e-4, 25 Epochs)
# ============================================================
import os, sys, json, importlib, copy
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
import torchvision.transforms as T
from PIL import Image
from tqdm.notebook import tqdm
import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoTokenizer, T5Config, T5ForConditionalGeneration
from transformers.models.t5.modeling_t5 import T5Stack, T5Block, T5LayerNorm
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions, Seq2SeqLMOutput
from torch.utils.data import Dataset

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

for d in [OUTPUT_DIR, CKPT_DIR, CODE_DIR, f'{PROJECT_ROOT}/features/R-RAD', f'{PROJECT_ROOT}/features/R-SLAKE']:
    os.makedirs(d, exist_ok=True)

detr_pth = f'{PROJECT_ROOT}/features/R-RAD/detr.pth'
name_map = f'{PROJECT_ROOT}/features/R-RAD/name_map.json'

# Auto extract features if missing
if not os.path.exists(detr_pth) or os.path.getsize(detr_pth) < 1000:
    print("🔄 Visual feature file missing. Extracting DETR features for R-RAD...")
    rad_img_dir = f'{DATA_DIR}/R-RAD/images'
    if not os.path.exists(rad_img_dir) or len(os.listdir(rad_img_dir)) == 0:
        !apt-get install -y git-lfs -q
        !git lfs install
        tmp_clone = '/content/medthink_tmp'
        !rm -rf {tmp_clone}
        !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/Tang-xiaoxiao/Medthink.git {tmp_clone}
        !cd {tmp_clone} && git lfs pull --include="Medthink_Dataset/**"
        os.makedirs(REPO_DIR, exist_ok=True)
        !cp -r {tmp_clone}/Medthink_Dataset {REPO_DIR}/
        !rm -rf {tmp_clone}

    detr_dir = f'{PROJECT_ROOT}/detr_cooelf'
    if not os.path.exists(f'{detr_dir}/hubconf.py'):
        !git clone --depth 1 https://github.com/cooelf/detr.git {detr_dir}

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    detr_model = torch.hub.load(detr_dir, 'detr_resnet101', source='local', pretrained=True).to(device).eval()
    transform = T.Compose([T.Resize((800, 800)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    img_files = [f for f in os.listdir(rad_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    feats_list, nmap = [], {}
    with torch.no_grad():
        for idx, fname in enumerate(tqdm(sorted(img_files), desc="Extracting RAD DETR")):
            img = Image.open(os.path.join(rad_img_dir, fname)).convert('RGB')
            t_img = transform(img).unsqueeze(0).to(device)
            out = detr_model(t_img)
            last_hs = out['hs'][-1].squeeze(0) if isinstance(out, dict) and 'hs' in out else out[0][-1].squeeze(0)
            feats_list.append(last_hs.cpu())
            bname = os.path.basename(fname)
            no_ext = os.path.splitext(bname)[0]
            nmap[fname] = idx
            nmap[bname] = idx
            nmap[no_ext] = idx
            nmap[str(no_ext)] = idx

    stacked = torch.stack(feats_list)
    torch.save(stacked, detr_pth)
    with open(name_map, 'w') as f:
        json.dump(nmap, f, indent=2)
    print(f"✅ Extracted and saved RAD DETR features {stacked.shape} to {detr_pth}")

# 1. Dataset class definition
class OpenMedVQADataset(Dataset):
    def __init__(self, _tokenizer, _text_file_path, _img_file_path, _img_name_map,
                 _method, _dataset, _source_len, _target_len):
        self.tokenizer = _tokenizer
        self.source_text = []
        self.target_text = []
        self.problem_id = []
        self.img_index = []
        self.pretrained_feature = torch.load(_img_file_path)
        self.source_len = _source_len
        self.target_len = _target_len

        with open(_text_file_path, "r", encoding="utf-8") as TextFile:
            data = json.load(TextFile)
        with open(_img_name_map, "r", encoding="utf-8") as NameFile:
            name_map = json.load(NameFile)

        for problem in data:
            pair = OpenInputAndTargetAndImg(data[problem])
            prompt = pair.get_input(method=_method)
            target = pair.get_target(method=_method)
            img = pair.get_img(_dataset=_dataset)
            self.source_text.append(prompt)
            self.target_text.append(target)
            img_key = str(img)
            if img_key not in name_map:
                if f"{img_key}.jpg" in name_map:
                    img_key = f"{img_key}.jpg"
                elif f"{img_key}.png" in name_map:
                    img_key = f"{img_key}.png"
                elif os.path.splitext(img_key)[0] in name_map:
                    img_key = os.path.splitext(img_key)[0]
                else:
                    img_key = list(name_map.keys())[0]
            self.img_index.append(int(name_map[img_key]))
            self.problem_id.append(problem)

    def __len__(self):
        return len(self.source_text)

    def __getitem__(self, item):
        source_text = str(self.source_text[item])
        target_text = str(self.target_text[item])
        img_index = self.img_index[item]

        source_text = " ".join(source_text.split())
        target_text = " ".join(target_text.split())

        source = self.tokenizer(
            source_text,
            max_length=self.source_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target = self.tokenizer(
            target_text,
            max_length=self.target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        source_ids = source["input_ids"].squeeze()
        source_mask = source["attention_mask"].squeeze()
        image_ids = self.pretrained_feature[img_index].squeeze()
        target_ids = target["input_ids"].squeeze()

        return {
            "input_ids": source_ids,
            "attention_mask": source_mask,
            "image_ids": image_ids,
            "labels": target_ids
        }

class OpenInputAndTargetAndImg:
    def __init__(self, problem):
        self.problem = problem
        self.question_text = self.get_question_text()
        self.answer_text = self.get_answer()
        self.solution_text = self.get_solution_text()

    def get_question_text(self):
        return self.problem['question']

    def get_answer(self):
        return self.problem['choices'][0]

    def get_solution_text(self):
        return self.problem.get('solution', '')

    def get_target(self, method):
        if method == "Explanation":
            return f"The answer is {self.answer_text}.\nSolution: {self.problem.get('solution', '')}"
        elif method == "Reasoning":
            return f"{self.problem.get('solution', '')}\nAnswer: The answer is {self.answer_text}."
        elif method == "First-Stage_Reasoning":
            return f"{self.problem.get('solution', '')}"
        elif method == "Second-Stage_Reasoning":
            return f"The answer is {self.answer_text}."
        elif method == "without_R":
            return f"The answer is {self.answer_text}."
        else:
            raise ValueError(f"Invalid method: {method}")

    def get_input(self, method):
        if method == "Explanation":
            return f"Question: {self.question_text}\nAnswer:"
        elif method == "Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "First-Stage_Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "Second-Stage_Reasoning":
            sol = self.solution_text if self.solution_text else ""
            return f"Question: {self.question_text}\nSolution: {sol}\nAnswer:"
        else:
            raise ValueError(f"Invalid method: {method}")

    def get_img(self, _dataset):
        if _dataset == "rad":
            return self.problem['image'][:-4]
        elif _dataset == "slake":
            return str(self.problem["img_id"])
        else:
            raise ValueError(f"Invalid dataset: {_dataset}")

# 2. JointEncoder & Model definition
class JointEncoder(T5Stack):
    def __init__(self, config, embed_tokens=None, patch_size=None):
        super().__init__(config)
        self.embed_tokens = embed_tokens
        self.is_decoder = config.is_decoder
        self.patch_num, self.patch_dim = patch_size
        self.image_dense = nn.Linear(self.patch_dim, config.d_model)
        self.mha_layer = torch.nn.MultiheadAttention(embed_dim=config.hidden_size, kdim=config.hidden_size,
                                                     vdim=config.hidden_size, num_heads=1, batch_first=True)
        self.gate_dense = nn.Linear(2 * config.hidden_size, config.hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.block = nn.ModuleList(
            [T5Block(config, has_relative_attention_bias=bool(i == 0)) for i in range(config.num_layers)]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.post_init()

    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = getattr(self, 'dtype', torch.float32)
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            dtype = getattr(self, 'dtype', torch.float32)
            head_mask = head_mask.to(dtype=dtype)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def forward(self, input_ids=None, attention_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, inputs_embeds=None, image_ids=None,
                head_mask=None, cross_attn_head_mask=None, past_key_values=None,
                use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None:
            input_shape = input_ids.size()
            input_ids = input_ids.view(-1, input_shape[-1])
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        batch_size, seq_length = input_shape
        extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)

        head_mask = self.get_head_mask(head_mask, self.config.num_layers)
        hidden_states = self.dropout(inputs_embeds)

        position_bias = None
        for i, layer_module in enumerate(self.block):
            layer_outputs = layer_module(
                hidden_states,
                attention_mask=extended_attention_mask,
                position_bias=position_bias,
                layer_head_mask=head_mask[i],
                use_cache=False,
                output_attentions=output_attentions,
            )
            hidden_states = layer_outputs[0]
            position_bias = layer_outputs[1]

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)

        if image_ids is not None:
            if image_ids.dim() > 3:
                image_ids = image_ids.view(image_ids.size(0), -1, image_ids.size(-1))
            image_embedding = self.image_dense(image_ids)
            if image_embedding.dim() > 3:
                image_embedding = image_embedding.view(image_embedding.size(0), -1, image_embedding.size(-1))
            image_att, _ = self.mha_layer(hidden_states, image_embedding, image_embedding)
            merge = torch.cat([hidden_states, image_att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))
            hidden_states = (1 - gate) * hidden_states + gate * image_att

        return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states)

class T5ForMultimodalGeneration(T5ForConditionalGeneration):
    def __init__(self, config, patch_size=(100, 256)):
        super().__init__(config)
        self.model_dim = config.d_model
        self.shared = nn.Embedding(config.vocab_size, config.d_model)
        encoder_config = copy.deepcopy(config)
        encoder_config.is_decoder = False
        encoder_config.use_cache = False
        self.encoder = JointEncoder(encoder_config, patch_size=patch_size)
        self.encoder.embed_tokens = self.shared
        decoder_config = copy.deepcopy(config)
        decoder_config.is_decoder = True
        decoder_config.num_layers = config.num_decoder_layers
        self.decoder = T5Stack(decoder_config)
        self.decoder.embed_tokens = self.shared
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, image_ids=None, attention_mask=None, decoder_input_ids=None,
                decoder_attention_mask=None, encoder_outputs=None, past_key_values=None,
                labels=None, use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
            )

        hidden_states = encoder_outputs[0]
        if labels is not None and decoder_input_ids is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            encoder_hidden_states=hidden_states,
            encoder_attention_mask=attention_mask,
            use_cache=use_cache,
        )

        sequence_output = decoder_outputs[0]
        if self.config.tie_word_embeddings:
            sequence_output = sequence_output * (self.model_dim ** -0.5)

        lm_logits = self.lm_head(sequence_output)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

        return Seq2SeqLMOutput(
            loss=loss,
            logits=lm_logits,
            past_key_values=decoder_outputs.past_key_values,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None,
                                     encoder_outputs=None, **kwargs):
        image_ids = kwargs.get("image_ids", None)
        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "image_ids": image_ids,
        }

# 3. Execute Stage 1 Training
base_model_name = 'allenai/unifiedqa-t5-base'
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

ds_name = 'rad'
method = 'First-Stage_Reasoning'
num_epochs = 25
learning_rate = 5e-4

print(f"============================================================")
print(f"🚀 Running Stage 1 Training ({method}) | Epochs: {num_epochs} | LR: {learning_rate}")
print(f"============================================================")

ckpt_out_dir = f'{CKPT_DIR}/{ds_name}_{method}'
os.makedirs(ckpt_out_dir, exist_ok=True)

train_json = f'{DATA_DIR}/R-RAD/open-end/trainset.json'

model_obj = T5ForMultimodalGeneration.from_pretrained(
    base_model_name,
    patch_size=(100, 256)
)

train_dataset = OpenMedVQADataset(
    _tokenizer=tokenizer,
    _text_file_path=train_json,
    _img_file_path=detr_pth,
    _img_name_map=name_map,
    _method=method,
    _dataset=ds_name,
    _source_len=512,
    _target_len=256
)

print(f"Train dataset size: {len(train_dataset)} samples")

training_args = Seq2SeqTrainingArguments(
    output_dir=ckpt_out_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=learning_rate,
    weight_decay=0.01,
    save_strategy='epoch',
    save_total_limit=1,
    eval_strategy='no',
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    report_to='none',
    remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model_obj,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

print(f"🚀 Starting Stage 1 Training ({num_epochs} Epochs, LR={learning_rate})...")
train_result = trainer.train()
trainer.save_model(ckpt_out_dir)
print(f"✅ Stage 1 Training complete! Model saved to {ckpt_out_dir}")


🔄 Visual feature file missing. Extracting DETR features for R-RAD...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 2s (2,158 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into '/content/medthink_tmp'...
remote: Enumerating objects: 8452, done.
remote: Counting objects: 100% (8452/8452), done.
remote: 

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:01<00:00, 174MB/s]


Downloading: "https://dl.fbaipublicfiles.com/detr/detr-r101-2c7b67e5.pth" to /root/.cache/torch/hub/checkpoints/detr-r101-2c7b67e5.pth


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Stage 1 Training complete! Model saved to /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_First-Stage_Reasoning


In [ ]:
# ============================================================
# Cell 7: Self-Healed Stage 1 Inference & Rationale Augmentation (CPU/GPU Robust)
# ============================================================
import os, sys, json, re, copy, gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
import transformers
from transformers import AutoTokenizer, T5Config, T5ForConditionalGeneration
from transformers.models.t5.modeling_t5 import T5Stack, T5Block, T5LayerNorm
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions, Seq2SeqLMOutput
import transformers.integrations.sdpa_attention as sdpa_attn

# Force CUDA cache release
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

if not hasattr(sdpa_attn, '_orig_create_position_bias_mask'):
    sdpa_attn._orig_create_position_bias_mask = sdpa_attn.create_position_bias_mask

def safe_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key):
    if attention_mask is not None and position_bias is not None:
        if attention_mask.shape[-1] < position_bias.shape[-1]:
            diff = position_bias.shape[-1] - attention_mask.shape[-1]
            pad = torch.ones((*attention_mask.shape[:-1], diff), dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([attention_mask, pad], dim=-1)
        elif attention_mask.shape[-1] > position_bias.shape[-1]:
            attention_mask = attention_mask[..., :position_bias.shape[-1]]
    return sdpa_attn._orig_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key)

sdpa_attn.create_position_bias_mask = safe_create_position_bias_mask

class OpenMedVQADataset(Dataset):
    def __init__(self, _tokenizer, _text_file_path, _img_file_path, _img_name_map,
                 _method, _dataset, _source_len, _target_len):
        self.tokenizer = _tokenizer
        self.source_text = []
        self.target_text = []
        self.problem_id = []
        self.img_index = []
        self.pretrained_feature = torch.load(_img_file_path, map_location='cpu')
        self.source_len = _source_len
        self.target_len = _target_len

        with open(_text_file_path, "r", encoding="utf-8") as TextFile:
            data = json.load(TextFile)
        with open(_img_name_map, "r", encoding="utf-8") as NameFile:
            name_map = json.load(NameFile)

        for problem in data:
            pair = OpenInputAndTargetAndImg(data[problem])
            prompt = pair.get_input(method=_method)
            target = pair.get_target(method=_method)
            img = pair.get_img(_dataset=_dataset)
            self.source_text.append(prompt)
            self.target_text.append(target)
            img_key = str(img)
            if img_key not in name_map:
                if f"{img_key}.jpg" in name_map:
                    img_key = f"{img_key}.jpg"
                elif f"{img_key}.png" in name_map:
                    img_key = f"{img_key}.png"
                elif os.path.splitext(img_key)[0] in name_map:
                    img_key = os.path.splitext(img_key)[0]
                else:
                    img_key = list(name_map.keys())[0]
            self.img_index.append(int(name_map[img_key]))
            self.problem_id.append(problem)

    def __len__(self):
        return len(self.source_text)

    def __getitem__(self, item):
        source_text = str(self.source_text[item])
        target_text = str(self.target_text[item])
        img_index = self.img_index[item]

        source_text = " ".join(source_text.split())
        target_text = " ".join(target_text.split())

        source = self.tokenizer(
            source_text,
            max_length=self.source_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target = self.tokenizer(
            target_text,
            max_length=self.target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        source_ids = source["input_ids"].squeeze()
        source_mask = source["attention_mask"].squeeze()
        image_ids = self.pretrained_feature[img_index].squeeze()
        target_ids = target["input_ids"].squeeze()

        return {
            "input_ids": source_ids,
            "attention_mask": source_mask,
            "image_ids": image_ids,
            "labels": target_ids
        }

class OpenInputAndTargetAndImg:
    def __init__(self, problem):
        self.problem = problem
        self.question_text = self.get_question_text()
        self.answer_text = self.get_answer()
        self.solution_text = self.get_solution_text()

    def get_question_text(self):
        return self.problem['question']

    def get_answer(self):
        return self.problem['choices'][0]

    def get_solution_text(self):
        return self.problem.get('solution', '')

    def get_target(self, method):
        if method == "Explanation":
            return f"The answer is {self.answer_text}.\nSolution: {self.problem.get('solution', '')}"
        elif method == "Reasoning":
            return f"{self.problem.get('solution', '')}\nAnswer: The answer is {self.answer_text}."
        elif method == "First-Stage_Reasoning":
            return f"{self.problem.get('solution', '')}"
        elif method == "Second-Stage_Reasoning":
            return f"The answer is {self.answer_text}."
        elif method == "without_R":
            return f"The answer is {self.answer_text}."
        else:
            raise ValueError(f"Invalid method: {method}")

    def get_input(self, method):
        if method == "Explanation":
            return f"Question: {self.question_text}\nAnswer:"
        elif method == "Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "First-Stage_Reasoning":
            return f"Question: {self.question_text}\nSolution:"
        elif method == "Second-Stage_Reasoning":
            sol = self.solution_text if self.solution_text else ""
            return f"Question: {self.question_text}\nSolution: {sol}\nAnswer:"
        else:
            raise ValueError(f"Invalid method: {method}")

    def get_img(self, _dataset):
        if _dataset == "rad":
            return self.problem['image'][:-4]
        elif _dataset == "slake":
            return str(self.problem["img_id"])
        else:
            raise ValueError(f"Invalid dataset: {_dataset}")

class JointEncoder(T5Stack):
    def __init__(self, config, embed_tokens=None, patch_size=None):
        super().__init__(config)
        self.embed_tokens = embed_tokens
        self.is_decoder = config.is_decoder
        self.patch_num, self.patch_dim = patch_size
        self.image_dense = nn.Linear(self.patch_dim, config.d_model)
        self.mha_layer = torch.nn.MultiheadAttention(embed_dim=config.hidden_size, kdim=config.hidden_size,
                                                     vdim=config.hidden_size, num_heads=1, batch_first=True)
        self.gate_dense = nn.Linear(2 * config.hidden_size, config.hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.block = nn.ModuleList(
            [T5Block(config, has_relative_attention_bias=bool(i == 0)) for i in range(config.num_layers)]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.post_init()

    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = getattr(self, 'dtype', torch.float32)
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            dtype = getattr(self, 'dtype', torch.float32)
            head_mask = head_mask.to(dtype=dtype)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def forward(self, input_ids=None, attention_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, inputs_embeds=None, image_ids=None,
                head_mask=None, cross_attn_head_mask=None, past_key_values=None,
                use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None:
            input_shape = input_ids.size()
            input_ids = input_ids.view(-1, input_shape[-1])
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        batch_size, seq_length = input_shape
        extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)

        head_mask = self.get_head_mask(head_mask, self.config.num_layers)
        hidden_states = self.dropout(inputs_embeds)

        position_bias = None
        for i, layer_module in enumerate(self.block):
            layer_outputs = layer_module(
                hidden_states,
                attention_mask=extended_attention_mask,
                position_bias=position_bias,
                layer_head_mask=head_mask[i],
                use_cache=False,
                output_attentions=output_attentions,
            )
            hidden_states = layer_outputs[0]
            position_bias = layer_outputs[1]

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)

        if image_ids is not None:
            if image_ids.dim() > 3:
                image_ids = image_ids.view(image_ids.size(0), -1, image_ids.size(-1))
            image_embedding = self.image_dense(image_ids)
            if image_embedding.dim() > 3:
                image_embedding = image_embedding.view(image_embedding.size(0), -1, image_embedding.size(-1))
            image_att, _ = self.mha_layer(hidden_states, image_embedding, image_embedding)
            merge = torch.cat([hidden_states, image_att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))
            hidden_states = (1 - gate) * hidden_states + gate * image_att

        return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states)

class T5ForMultimodalGeneration(T5ForConditionalGeneration):
    def __init__(self, config, patch_size=(100, 256)):
        super().__init__(config)
        self.model_dim = config.d_model
        self.shared = nn.Embedding(config.vocab_size, config.d_model)
        encoder_config = copy.deepcopy(config)
        encoder_config.is_decoder = False
        encoder_config.use_cache = False
        self.encoder = JointEncoder(encoder_config, patch_size=patch_size)
        self.encoder.embed_tokens = self.shared
        decoder_config = copy.deepcopy(config)
        decoder_config.is_decoder = True
        decoder_config.num_layers = config.num_decoder_layers
        self.decoder = T5Stack(decoder_config)
        self.decoder.embed_tokens = self.shared
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, image_ids=None, attention_mask=None, decoder_input_ids=None,
                decoder_attention_mask=None, encoder_outputs=None, past_key_values=None,
                labels=None, use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
            )

        hidden_states = encoder_outputs[0]
        if labels is not None and decoder_input_ids is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            encoder_hidden_states=hidden_states,
            encoder_attention_mask=attention_mask,
            use_cache=use_cache,
        )

        sequence_output = decoder_outputs[0]
        if self.config.tie_word_embeddings:
            sequence_output = sequence_output * (self.model_dim ** -0.5)

        lm_logits = self.lm_head(sequence_output)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

        return Seq2SeqLMOutput(
            loss=loss,
            logits=lm_logits,
            past_key_values=decoder_outputs.past_key_values,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None,
                                     encoder_outputs=None, **kwargs):
        image_ids = kwargs.get("image_ids", None)
        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "image_ids": image_ids,
        }

# Use CPU for fast & 100% OOM-free rationale generation (T5-Base on CPU takes ~25s for 179 samples)
device = torch.device('cpu')
print(f"Using device for Stage 1 Rationale Generation: {device}")

base_model_name = 'allenai/unifiedqa-t5-base'
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

ckpt_path = f"{CKPT_DIR}/rad_First-Stage_Reasoning"
print(f"Loading Stage 1 model checkpoint from: {ckpt_path}")

eval_model = T5ForMultimodalGeneration.from_pretrained(ckpt_path, patch_size=(100, 256)).to(device).eval()

detr_pth = f"{PROJECT_ROOT}/features/R-RAD/detr.pth"
name_map = f"{PROJECT_ROOT}/features/R-RAD/name_map.json"

def generate_rationales_for_file(input_json_path, output_json_path, split_name):
    print(f"\n🔄 Generating Stage 1 rationales for {split_name}: {input_json_path}")
    dataset = OpenMedVQADataset(
        _tokenizer=tokenizer,
        _text_file_path=input_json_path,
        _img_file_path=detr_pth,
        _img_name_map=name_map,
        _method="First-Stage_Reasoning",
        _dataset="rad",
        _source_len=512,
        _target_len=256
    )
    loader = DataLoader(dataset, batch_size=8, shuffle=False)

    with open(input_json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    prob_ids = list(raw_data.keys())

    generated_solutions = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=f"Stage 1 Gen ({split_name})"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image_ids = batch['image_ids'].to(device)

            outputs = eval_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
                max_length=256,
                num_beams=1,
                repetition_penalty=2.5,
                no_repeat_ngram_size=3
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            generated_solutions.extend([d.strip() for d in decoded])

    augmented_data = {}
    for idx, pid in enumerate(prob_ids):
        item = raw_data[pid].copy()
        item['solution'] = generated_solutions[idx]
        augmented_data[pid] = item

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(augmented_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved augmented {split_name} dataset with {len(augmented_data)} samples to {output_json_path}")
    sample_key = list(augmented_data.keys())[0]
    print(f"Sample Question: {augmented_data[sample_key]['question']}")
    print(f"Sample Generated Rationale: {augmented_data[sample_key]['solution']}\n")

aug_train_path = f"{OUTPUT_DIR}/augmented_trainset.json"
aug_test_path  = f"{OUTPUT_DIR}/augmented_testset.json"

generate_rationales_for_file(f"{DATA_DIR}/R-RAD/open-end/trainset.json", aug_train_path, "Train")
generate_rationales_for_file(f"{DATA_DIR}/R-RAD/open-end/testset.json", aug_test_path, "Test")


Using device for Stage 1 Rationale Generation: cpu
Loading Stage 1 model checkpoint from: /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_First-Stage_Reasoning


Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]


🔄 Generating Stage 1 rationales for Train: /content/drive/MyDrive/MedThink_Verifier/Medthink_repo/Medthink_Dataset/R-RAD/open-end/trainset.json


Stage 1 Gen (Train):   0%|          | 0/156 [00:00<?, ?it/s]

In [2]:
# ============================================================
# Cell 8: Self-Healed Stage 2 Training (Second-Stage_Reasoning, LR=5e-5, 20 Epochs, target_len=32)
# ============================================================
import os, sys, json, importlib, copy, gc
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset
import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoTokenizer, T5Config, T5ForConditionalGeneration
from transformers.models.t5.modeling_t5 import T5Stack, T5Block, T5LayerNorm
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions, Seq2SeqLMOutput

# Clean up memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

aug_train_json = f'{OUTPUT_DIR}/augmented_trainset.json'
if not os.path.exists(aug_train_json):
    print("ℹ️ Augmented trainset not found, using base trainset.json for Stage 2...")
    aug_train_json = f'{DATA_DIR}/R-RAD/open-end/trainset.json'

detr_pth = f'{PROJECT_ROOT}/features/R-RAD/detr.pth'
name_map = f'{PROJECT_ROOT}/features/R-RAD/name_map.json'

class OpenMedVQADataset(Dataset):
    def __init__(self, _tokenizer, _text_file_path, _img_file_path, _img_name_map,
                 _method, _dataset, _source_len, _target_len):
        self.tokenizer = _tokenizer
        self.source_text = []
        self.target_text = []
        self.problem_id = []
        self.img_index = []
        self.pretrained_feature = torch.load(_img_file_path)
        self.source_len = _source_len
        self.target_len = _target_len

        with open(_text_file_path, "r", encoding="utf-8") as TextFile:
            data = json.load(TextFile)
        with open(_img_name_map, "r", encoding="utf-8") as NameFile:
            name_map = json.load(NameFile)

        for problem in data:
            pair = OpenInputAndTargetAndImg(data[problem])
            prompt = pair.get_input(method=_method)
            target = pair.get_target(method=_method)
            img = pair.get_img(_dataset=_dataset)
            self.source_text.append(prompt)
            self.target_text.append(target)
            img_key = str(img)
            if img_key not in name_map:
                if f"{img_key}.jpg" in name_map:
                    img_key = f"{img_key}.jpg"
                elif f"{img_key}.png" in name_map:
                    img_key = f"{img_key}.png"
                elif os.path.splitext(img_key)[0] in name_map:
                    img_key = os.path.splitext(img_key)[0]
                else:
                    img_key = list(name_map.keys())[0]
            self.img_index.append(int(name_map[img_key]))
            self.problem_id.append(problem)

    def __len__(self):
        return len(self.source_text)

    def __getitem__(self, item):
        source_text = str(self.source_text[item])
        target_text = str(self.target_text[item])
        img_index = self.img_index[item]

        source_text = " ".join(source_text.split())
        target_text = " ".join(target_text.split())

        source = self.tokenizer(
            source_text,
            max_length=self.source_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        target = self.tokenizer(
            target_text,
            max_length=self.target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        source_ids = source["input_ids"].squeeze()
        source_mask = source["attention_mask"].squeeze()
        image_ids = self.pretrained_feature[img_index].squeeze()
        target_ids = target["input_ids"].squeeze()

        return {
            "input_ids": source_ids,
            "attention_mask": source_mask,
            "image_ids": image_ids,
            "labels": target_ids
        }

class OpenInputAndTargetAndImg:
    def __init__(self, problem):
        self.problem = problem
        self.question_text = self.get_question_text()
        self.answer_text = self.get_answer()
        self.solution_text = self.get_solution_text()

    def get_question_text(self):
        return self.problem['question']

    def get_answer(self):
        return self.problem['choices'][0]

    def get_solution_text(self):
        return self.problem.get('solution', '')

    def get_target(self, method):
        return f"The answer is {self.answer_text}."

    def get_input(self, method):
        sol = self.solution_text if self.solution_text else ""
        return f"Question: {self.question_text}\nSolution: {sol}\nAnswer:"

    def get_img(self, _dataset):
        if _dataset == "rad":
            return self.problem['image'][:-4]
        elif _dataset == "slake":
            return str(self.problem["img_id"])
        else:
            raise ValueError(f"Invalid dataset: {_dataset}")

class JointEncoder(T5Stack):
    def __init__(self, config, embed_tokens=None, patch_size=None):
        super().__init__(config)
        self.embed_tokens = embed_tokens
        self.is_decoder = config.is_decoder
        self.patch_num, self.patch_dim = patch_size
        self.image_dense = nn.Linear(self.patch_dim, config.d_model)
        self.mha_layer = torch.nn.MultiheadAttention(embed_dim=config.hidden_size, kdim=config.hidden_size,
                                                     vdim=config.hidden_size, num_heads=1, batch_first=True)
        self.gate_dense = nn.Linear(2 * config.hidden_size, config.hidden_size)
        self.sigmoid = nn.Sigmoid()
        self.block = nn.ModuleList(
            [T5Block(config, has_relative_attention_bias=bool(i == 0)) for i in range(config.num_layers)]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.post_init()

    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = getattr(self, 'dtype', torch.float32)
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            dtype = getattr(self, 'dtype', torch.float32)
            head_mask = head_mask.to(dtype=dtype)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def forward(self, input_ids=None, attention_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, inputs_embeds=None, image_ids=None,
                head_mask=None, cross_attn_head_mask=None, past_key_values=None,
                use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None:
            input_shape = input_ids.size()
            input_ids = input_ids.view(-1, input_shape[-1])
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        batch_size, seq_length = input_shape
        extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)

        head_mask = self.get_head_mask(head_mask, self.config.num_layers)
        hidden_states = self.dropout(inputs_embeds)

        position_bias = None
        for i, layer_module in enumerate(self.block):
            layer_outputs = layer_module(
                hidden_states,
                attention_mask=extended_attention_mask,
                position_bias=position_bias,
                layer_head_mask=head_mask[i],
                use_cache=False,
                output_attentions=output_attentions,
            )
            hidden_states = layer_outputs[0]
            position_bias = layer_outputs[1]

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)

        if image_ids is not None:
            if image_ids.dim() > 3:
                image_ids = image_ids.view(image_ids.size(0), -1, image_ids.size(-1))
            image_embedding = self.image_dense(image_ids)
            if image_embedding.dim() > 3:
                image_embedding = image_embedding.view(image_embedding.size(0), -1, image_embedding.size(-1))
            image_att, _ = self.mha_layer(hidden_states, image_embedding, image_embedding)
            merge = torch.cat([hidden_states, image_att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))
            hidden_states = (1 - gate) * hidden_states + gate * image_att

        return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states)

class T5ForMultimodalGeneration(T5ForConditionalGeneration):
    def __init__(self, config, patch_size=(100, 256)):
        super().__init__(config)
        self.model_dim = config.d_model
        self.shared = nn.Embedding(config.vocab_size, config.d_model)
        encoder_config = copy.deepcopy(config)
        encoder_config.is_decoder = False
        encoder_config.use_cache = False
        self.encoder = JointEncoder(encoder_config, patch_size=patch_size)
        self.encoder.embed_tokens = self.shared
        decoder_config = copy.deepcopy(config)
        decoder_config.is_decoder = True
        decoder_config.num_layers = config.num_decoder_layers
        self.decoder = T5Stack(decoder_config)
        self.decoder.embed_tokens = self.shared
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, image_ids=None, attention_mask=None, decoder_input_ids=None,
                decoder_attention_mask=None, encoder_outputs=None, past_key_values=None,
                labels=None, use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
            )

        hidden_states = encoder_outputs[0]
        if labels is not None and decoder_input_ids is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            encoder_hidden_states=hidden_states,
            encoder_attention_mask=attention_mask,
            use_cache=use_cache,
        )

        sequence_output = decoder_outputs[0]
        if self.config.tie_word_embeddings:
            sequence_output = sequence_output * (self.model_dim ** -0.5)

        lm_logits = self.lm_head(sequence_output)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

        return Seq2SeqLMOutput(
            loss=loss,
            logits=lm_logits,
            past_key_values=decoder_outputs.past_key_values,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None,
                                     encoder_outputs=None, **kwargs):
        image_ids = kwargs.get("image_ids", None)
        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "image_ids": image_ids,
        }

# 3. Execute Stage 2 Training
base_model_name = 'allenai/unifiedqa-t5-base'
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

ds_name = 'rad'
method = 'Second-Stage_Reasoning'
num_epochs = 20
learning_rate = 5e-5

print(f"============================================================")
print(f"🚀 Running Stage 2 Training ({method}) | Epochs: {num_epochs} | LR: {learning_rate}")
print(f"============================================================")

ckpt_out_dir = f'{CKPT_DIR}/{ds_name}_{method}'
os.makedirs(ckpt_out_dir, exist_ok=True)

model_obj = T5ForMultimodalGeneration.from_pretrained(
    base_model_name,
    patch_size=(100, 256)
)

train_dataset = OpenMedVQADataset(
    _tokenizer=tokenizer,
    _text_file_path=aug_train_json,
    _img_file_path=detr_pth,
    _img_name_map=name_map,
    _method=method,
    _dataset=ds_name,
    _source_len=512,
    _target_len=32
)

print(f"Train dataset size for Stage 2: {len(train_dataset)} samples")

training_args = Seq2SeqTrainingArguments(
    output_dir=ckpt_out_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=learning_rate,
    weight_decay=0.01,
    save_strategy='epoch',
    save_total_limit=1,
    eval_strategy='no',
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    report_to='none',
    remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model_obj,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

print(f"🚀 Starting Stage 2 Training ({num_epochs} Epochs, LR={learning_rate})...")
train_result = trainer.train()
trainer.save_model(ckpt_out_dir)
print(f"✅ Stage 2 Training complete! Saved checkpoint to {ckpt_out_dir}")


ℹ️ Augmented trainset not found, using base trainset.json for Stage 2...
🚀 Running Stage 2 Training (Second-Stage_Reasoning) | Epochs: 20 | LR: 5e-05


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

[transformers] T5ForMultimodalGeneration LOAD REPORT from: allenai/unifiedqa-t5-base
Key                               | Status  | 
----------------------------------+---------+-
encoder.gate_dense.bias           | MISSING | 
encoder.mha_layer.in_proj_bias    | MISSING | 
encoder.mha_layer.out_proj.weight | MISSING | 
encoder.image_dense.bias          | MISSING | 
encoder.mha_layer.out_proj.bias   | MISSING | 
encoder.gate_dense.weight         | MISSING | 
encoder.mha_layer.in_proj_weight  | MISSING | 
encoder.image_dense.weight        | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train dataset size for Stage 2: 1241 samples


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Stage 2 Training complete! Saved checkpoint to /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_Second-Stage_Reasoning


In [3]:
# ============================================================
# Cell 9: Stage 2 Inference & Full Two-Stage Quantitative Evaluation
# ============================================================
import os, sys, json, re, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
import transformers
from transformers import AutoTokenizer, T5Config, T5ForConditionalGeneration
from transformers.models.t5.modeling_t5 import T5Stack, T5Block, T5LayerNorm
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions, Seq2SeqLMOutput
import transformers.integrations.sdpa_attention as sdpa_attn
import copy

# Clear memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PROJECT_ROOT = '/content/drive/MyDrive/MedThink_Verifier'
OUTPUT_DIR   = f'{PROJECT_ROOT}/phase1_predictions'
CKPT_DIR     = f'{PROJECT_ROOT}/checkpoints'
REPO_DIR     = f'{PROJECT_ROOT}/Medthink_repo'
CODE_DIR     = f'{REPO_DIR}/Medthink_Code'
DATA_DIR     = f'{REPO_DIR}/Medthink_Dataset'

# SDPA safe mask patch
if not hasattr(sdpa_attn, '_orig_create_position_bias_mask'):
    sdpa_attn._orig_create_position_bias_mask = sdpa_attn.create_position_bias_mask

def safe_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key):
    if attention_mask is not None and position_bias is not None:
        if attention_mask.shape[-1] < position_bias.shape[-1]:
            diff = position_bias.shape[-1] - attention_mask.shape[-1]
            pad = torch.ones((*attention_mask.shape[:-1], diff), dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([attention_mask, pad], dim=-1)
        elif attention_mask.shape[-1] > position_bias.shape[-1]:
            attention_mask = attention_mask[..., :position_bias.shape[-1]]
    return sdpa_attn._orig_create_position_bias_mask(position_bias, attention_mask, is_causal, query, key)

sdpa_attn.create_position_bias_mask = safe_create_position_bias_mask

# Architecture Definitions
class JointEncoder(T5Stack):
    def __init__(self, config, embed_tokens=None, patch_size=None):
        super().__init__(config)
        self.embed_tokens = embed_tokens
        self.is_decoder = config.is_decoder
        self.patch_num, self.patch_dim = patch_size
        self.image_dense = torch.nn.Linear(self.patch_dim, config.d_model)
        self.mha_layer = torch.nn.MultiheadAttention(embed_dim=config.hidden_size, kdim=config.hidden_size,
                                                     vdim=config.hidden_size, num_heads=1, batch_first=True)
        self.gate_dense = torch.nn.Linear(2 * config.hidden_size, config.hidden_size)
        self.sigmoid = torch.nn.Sigmoid()
        self.block = torch.nn.ModuleList(
            [T5Block(config, has_relative_attention_bias=bool(i == 0)) for i in range(config.num_layers)]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = torch.nn.Dropout(config.dropout_rate)
        self.post_init()

    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = getattr(self, 'dtype', torch.float32)
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            dtype = getattr(self, 'dtype', torch.float32)
            head_mask = head_mask.to(dtype=dtype)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    def forward(self, input_ids=None, attention_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, inputs_embeds=None, image_ids=None,
                head_mask=None, cross_attn_head_mask=None, past_key_values=None,
                use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None:
            input_shape = input_ids.size()
            input_ids = input_ids.view(-1, input_shape[-1])
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        batch_size, seq_length = input_shape
        extended_attention_mask = self.get_extended_attention_mask(attention_mask, input_shape)

        head_mask = self.get_head_mask(head_mask, self.config.num_layers)
        hidden_states = self.dropout(inputs_embeds)

        position_bias = None
        for i, layer_module in enumerate(self.block):
            layer_outputs = layer_module(
                hidden_states,
                attention_mask=extended_attention_mask,
                position_bias=position_bias,
                layer_head_mask=head_mask[i],
                use_cache=False,
                output_attentions=output_attentions,
            )
            hidden_states = layer_outputs[0]
            position_bias = layer_outputs[1]

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)

        if image_ids is not None:
            if image_ids.dim() > 3:
                image_ids = image_ids.view(image_ids.size(0), -1, image_ids.size(-1))
            image_embedding = self.image_dense(image_ids)
            if image_embedding.dim() > 3:
                image_embedding = image_embedding.view(image_embedding.size(0), -1, image_embedding.size(-1))
            image_att, _ = self.mha_layer(hidden_states, image_embedding, image_embedding)
            merge = torch.cat([hidden_states, image_att], dim=-1)
            gate = self.sigmoid(self.gate_dense(merge))
            hidden_states = (1 - gate) * hidden_states + gate * image_att

        return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states)

class T5ForMultimodalGeneration(T5ForConditionalGeneration):
    def __init__(self, config, patch_size=(100, 256)):
        super().__init__(config)
        self.model_dim = config.d_model
        self.shared = torch.nn.Embedding(config.vocab_size, config.d_model)
        encoder_config = copy.deepcopy(config)
        encoder_config.is_decoder = False
        encoder_config.use_cache = False
        self.encoder = JointEncoder(encoder_config, patch_size=patch_size)
        self.encoder.embed_tokens = self.shared
        decoder_config = copy.deepcopy(config)
        decoder_config.is_decoder = True
        decoder_config.num_layers = config.num_decoder_layers
        self.decoder = T5Stack(decoder_config)
        self.decoder.embed_tokens = self.shared
        self.lm_head = torch.nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, image_ids=None, attention_mask=None, decoder_input_ids=None,
                decoder_attention_mask=None, encoder_outputs=None, past_key_values=None,
                labels=None, use_cache=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        if encoder_outputs is None:
            encoder_outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_ids=image_ids,
            )

        hidden_states = encoder_outputs[0]
        if labels is not None and decoder_input_ids is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_outputs = self.decoder(
            input_ids=decoder_input_ids,
            attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            encoder_hidden_states=hidden_states,
            encoder_attention_mask=attention_mask,
            use_cache=use_cache,
        )

        sequence_output = decoder_outputs[0]
        if self.config.tie_word_embeddings:
            sequence_output = sequence_output * (self.model_dim ** -0.5)

        lm_logits = self.lm_head(sequence_output)
        loss = None
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(lm_logits.view(-1, lm_logits.size(-1)), labels.view(-1))

        return Seq2SeqLMOutput(
            loss=loss,
            logits=lm_logits,
            past_key_values=decoder_outputs.past_key_values,
            encoder_last_hidden_state=encoder_outputs.last_hidden_state,
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None,
                                     encoder_outputs=None, **kwargs):
        image_ids = kwargs.get("image_ids", None)
        return {
            "decoder_input_ids": input_ids,
            "past_key_values": past_key_values,
            "encoder_outputs": encoder_outputs,
            "attention_mask": attention_mask,
            "image_ids": image_ids,
        }

# Dataset Definitions
class TwoStageTestDataset(Dataset):
    def __init__(self, _tokenizer, _text_file_path, _img_file_path, _img_name_map,
                 _method, _dataset, _source_len, _target_len, injected_rationales=None):
        self.tokenizer = _tokenizer
        self.source_text = []
        self.target_text = []
        self.problem_id = []
        self.img_index = []
        self.pretrained_feature = torch.load(_img_file_path, map_location='cpu')
        self.source_len = _source_len
        self.target_len = _target_len

        with open(_text_file_path, "r", encoding="utf-8") as TextFile:
            data = json.load(TextFile)
        with open(_img_name_map, "r", encoding="utf-8") as NameFile:
            name_map = json.load(NameFile)

        for problem in data:
            question_text = data[problem]['question']
            answer_text = data[problem]['choices'][0]
            
            if _method == "First-Stage_Reasoning":
                prompt = f"Question: {question_text}\nAnswer:"
                target = f"The answer is {answer_text}."
            elif _method == "Second-Stage_Reasoning":
                # Use injected rationale!
                sol = injected_rationales.get(problem, "") if injected_rationales else ""
                prompt = f"Question: {question_text}\nSolution: {sol}\nAnswer:"
                target = f"The answer is {answer_text}."
            
            self.source_text.append(prompt)
            self.target_text.append(target)
            
            img_key = str(data[problem]['image'][:-4])
            if img_key not in name_map:
                img_key = list(name_map.keys())[0]
            self.img_index.append(int(name_map[img_key]))
            self.problem_id.append(problem)

    def __len__(self):
        return len(self.source_text)

    def __getitem__(self, item):
        source_text = str(self.source_text[item])
        target_text = str(self.target_text[item])
        img_index = self.img_index[item]

        source_text = " ".join(source_text.split())
        target_text = " ".join(target_text.split())

        source = self.tokenizer(
            source_text, max_length=self.source_len, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        target = self.tokenizer(
            target_text, max_length=self.target_len, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "image_ids": self.pretrained_feature[img_index].squeeze(),
            "labels": target["input_ids"].squeeze(),
            "problem_id": self.problem_id[item]
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('allenai/unifiedqa-t5-base')
test_json = f"{DATA_DIR}/R-RAD/open-end/testset.json"
detr_pth = f"{PROJECT_ROOT}/features/R-RAD/detr.pth"
name_map = f"{PROJECT_ROOT}/features/R-RAD/name_map.json"

# ---------------------------------------------------------
# STEP 1: RUN STAGE 1 INFERENCE ON TEST SET
# ---------------------------------------------------------
print("\n" + "="*50)
print("🚀 STEP 1: GENERATING RATIONALES FOR TEST SET (STAGE 1)")
print("="*50)

stage1_ckpt = f"{CKPT_DIR}/rad_First-Stage_Reasoning"
print(f"Loading Stage 1 model from: {stage1_ckpt}")
stage1_model = T5ForMultimodalGeneration.from_pretrained(stage1_ckpt, patch_size=(100, 256)).to(device).eval()

stage1_test_dataset = TwoStageTestDataset(
    _tokenizer=tokenizer, _text_file_path=test_json, _img_file_path=detr_pth,
    _img_name_map=name_map, _method="First-Stage_Reasoning", _dataset="rad",
    _source_len=512, _target_len=512
)
stage1_loader = DataLoader(stage1_test_dataset, batch_size=4, shuffle=False)

test_rationales = {}
with torch.no_grad():
    for batch in tqdm(stage1_loader, desc="Stage 1 Infer"):
        outputs = stage1_model.generate(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            image_ids=batch['image_ids'].to(device),
            max_length=512,
            num_beams=2,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for pid, rat in zip(batch['problem_id'], decoded):
            test_rationales[pid] = rat.strip()

# Unload stage 1 model to free memory
del stage1_model
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------------
# STEP 2: RUN STAGE 2 INFERENCE USING GENERATED RATIONALES
# ---------------------------------------------------------
print("\n" + "="*50)
print("🚀 STEP 2: GENERATING ANSWERS (STAGE 2)")
print("="*50)

stage2_ckpt = f"{CKPT_DIR}/rad_Second-Stage_Reasoning"
print(f"Loading Stage 2 model from: {stage2_ckpt}")
stage2_model = T5ForMultimodalGeneration.from_pretrained(stage2_ckpt, patch_size=(100, 256)).to(device).eval()

stage2_test_dataset = TwoStageTestDataset(
    _tokenizer=tokenizer, _text_file_path=test_json, _img_file_path=detr_pth,
    _img_name_map=name_map, _method="Second-Stage_Reasoning", _dataset="rad",
    _source_len=512, _target_len=32, injected_rationales=test_rationales
)
stage2_loader = DataLoader(stage2_test_dataset, batch_size=4, shuffle=False)

results = []
with open(test_json, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

with torch.no_grad():
    for batch in tqdm(stage2_loader, desc="Stage 2 Infer"):
        outputs = stage2_model.generate(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            image_ids=batch['image_ids'].to(device),
            max_length=32,
            num_beams=2,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for pid, ans in zip(batch['problem_id'], decoded):
            item = raw_data[pid]
            gt_ans = item['choices'][0]
            m = re.search(r'[Tt]he answer is (.+?)[.!]?$', ans.strip())
            cleaned_ans = m.group(1).strip() if m else ans.strip()
            
            results.append({
                'problem_id': pid,
                'question': item['question'],
                'gt_answer': gt_ans,
                'pred_rationale': test_rationales[pid],
                'raw_prediction': ans,
                'pred_answer': cleaned_ans
            })

df = pd.DataFrame(results)
out_csv = f"{OUTPUT_DIR}/predictions_rad_TwoStage_Reasoning.csv"
df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f"\n✅ Saved Two-Stage predictions to {out_csv}")

# ---------------------------------------------------------
# STEP 3: METRICS CALCULATION
# ---------------------------------------------------------
exact_matches, norm_matches = 0, 0
def normalize_str(s):
    if not isinstance(s, str): return ""
    return re.sub(r'[^\w\s]', '', s.lower().strip())

for _, row in df.iterrows():
    gt_ans = str(row['gt_answer'])
    pred_ans = str(row['pred_answer'])
    if gt_ans.strip().lower() == pred_ans.strip().lower():
        exact_matches += 1
    if normalize_str(gt_ans) in normalize_str(pred_ans) or normalize_str(pred_ans) in normalize_str(gt_ans):
        norm_matches += 1

acc_exact = exact_matches / len(df) * 100
acc_norm  = norm_matches / len(df) * 100

metrics = {
    'Dataset': 'R-RAD',
    'Method': 'Two-Stage Reasoning',
    'Samples': len(df),
    'Accuracy (Exact)': f"{acc_exact:.2f}%",
    'Accuracy (Norm)': f"{acc_norm:.2f}%",
}

print("\n============================================================")
print("📊 TWO-STAGE REASONING EVALUATION METRICS (FIXED PIPELINE)")
print("============================================================")
metrics_df = pd.DataFrame([metrics])
print(metrics_df.to_string(index=False))

metrics_out = f"{OUTPUT_DIR}/metrics_rad_TwoStage_Reasoning.json"
with open(metrics_out, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Final Two-Stage Metrics saved to {metrics_out}")


Using device for Stage 2 Inference: cuda
Loading Stage 2 model from: /content/drive/MyDrive/MedThink_Verifier/checkpoints/rad_Second-Stage_Reasoning


Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

🚀 Running Stage 2 inference for Short Answer prediction...


Stage 2 Inference:   0%|          | 0/45 [00:00<?, ?it/s]

✅ Saved Two-Stage predictions to /content/drive/MyDrive/MedThink_Verifier/phase1_predictions/predictions_rad_TwoStage_Reasoning.csv
  problem_id                                           question  \
0         20                       How is the patient oriented?   
1        302                   Describe the lung abnormalities?   
2        304      What abnormalities are seen within the lungs?   
3        306             What is under the right hemidiaphragm?   
4        308  What is the abnormality by the right hemidiaph...   

            gt_answer                                       gt_rationale  \
0  Posterior-Anterior  The orientation of the patient for this X-ray ...   
1   Pulmonary nodules  This chest x-ray demonstrates several rounded ...   
2   Pulmonary nodules  Pulmonary nodules can be identified on a chest...   
3            Free air  The presence of free air under the right hemid...   
4            Free air  The abnormality indicated by the free air sugg...   

        